# 01 — Build historical dataset

In this notebook, we load all historical indicator files, clean them, and merge them into one country-year panel.

**Input:** files in `data/raw/`  
**Output:** `data/processed/historical_panel.csv`
**Variables used**

We combine several economic and development indicators:

- GDP (Purchase Price Parity). Used for GDP per capita  
- Population
- Human Development Index (HDI)
- Control of Corruption
- Employment in Agriculture sector
- Gini coefficient 
- Poverty rates ($3, $8.30, $10 per day)  

## Imports and paths

In [70]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
OUTPUTS_DIR = Path("../outputs")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## Helper function: load World Bank CSV

All World Bank CSVs share the same format: 4 header rows, then wide-format data 
(one column per year). This helper melts to long format and drops regional aggregates.

In [71]:
# World Bank aggregate codes to exclude (not real countries)
WB_AGGREGATE_CODES = {
    "AFE", "AFW", "ARB", "CEB", "CSS", "EAP", "EAR", "EAS", "ECA",
    "ECS", "EMU", "EUU", "FCS", "HIC", "HPC", "IBD", "IBT", "IDA",
    "IDB", "IDX", "INX", "LAC", "LCN", "LDC", "LIC", "LMC", "LMY",
    "LTE", "MEA", "MIC", "MNA", "NAC", "OED", "OSS", "PRE", "PSS",
    "PST", "SAS", "SSA", "SSF", "SST", "TEA", "TEC", "TLA", "TMN",
    "TSA", "TSS", "UMC", "WLD"
}

def load_wb_csv(filepath, var_name, year_start=1996, year_end=2023):
    """
    Load a World Bank CSV file and convert from wide to long format.
    Returns DataFrame with columns: country_name, country_code, year, <var_name>
    """
    df = pd.read_csv(filepath, skiprows=4)
    
    # Print indicator name for verification
    if "Indicator Name" in df.columns:
        print(f"  Indicator: {df['Indicator Name'].iloc[0]}")
    
    # Select year columns that exist in the file
    year_cols = [str(y) for y in range(year_start, year_end + 1) if str(y) in df.columns]
    keep_cols = ["Country Name", "Country Code"] + year_cols
    df = df[keep_cols]
    
    # Melt to long format
    df = df.melt(
        id_vars=["Country Name", "Country Code"],
        var_name="year",
        value_name=var_name
    )
    
    # Clean types
    df.rename(columns={"Country Name": "country_name", "Country Code": "country_code"}, inplace=True)
    df["year"] = df["year"].astype(int)
    df[var_name] = pd.to_numeric(df[var_name], errors="coerce")
    
    # Remove aggregate regions
    df = df[~df["country_code"].isin(WB_AGGREGATE_CODES)].reset_index(drop=True)
    
    print(f"  Shape: {df.shape}, Countries: {df['country_code'].nunique()}, "
          f"Non-null {var_name}: {df[var_name].notna().sum()}")
    return df

## Load GDP (PPP)

We use PPP-adjusted GDP rather than nominal GDP to match the units used in the SSP 
forecast data. This avoids systematically misscaling GPD for developing countries.

In [72]:
gdp = load_wb_csv(DATA_RAW / "WB_PPP_GDP_historical_new.csv", "gdp")
gdp.head()

  Indicator: GDP, PPP (current international $)
  Shape: (6076, 4), Countries: 217, Non-null gdp: 5516


,country_name,country_code,year,gdp
0,Aruba,ABW,1996,2.158919e+09
1,Afghanistan,AFG,1996,NaN
2,Angola,AGO,1996,4.735814e+10
3,Albania,ALB,1996,9.675284e+09
4,Andorra,AND,1996,1.698237e+09


## Load Control of Corruption

From the World Bank Worldwide Governance Indicators. Originally on a [-2.5, 2.5] scale — 
rescaled to [0, 100] for easier interpretation and consistency with the SSP forecasts.

In [73]:
corruption = load_wb_csv(
    DATA_RAW / "ControlOfCorruption_Historical_WorldBank_1996-2023.csv",
    "control_of_corruption"
)

# --- RESCALE: WB [-2.5, 2.5] → [0, 1] to match SSP Explorer forecast scale ---
#
# The World Bank Governance Indicators use a standardized score centered on 0,
# ranging roughly from -2.5 (most corrupt) to +2.5 (least corrupt).
#
# The SSP Extension Explorer forecasts use a normalized [0, 1] index for the
# same underlying concept (higher = less corrupt).
#
# To make the historical training data comparable to forecast features,
# we apply a min-max rescaling:
#   corruption_01 = (wb_score - wb_min) / (wb_max - wb_min)
#
# Using the theoretical WB bounds of [-2.5, 2.5]:
WB_CORR_MIN = -2.5
WB_CORR_MAX =  2.5
corruption["control_of_corruption"] = (
    (corruption["control_of_corruption"] - WB_CORR_MIN) / (WB_CORR_MAX - WB_CORR_MIN)
)

vals = corruption["control_of_corruption"].dropna()
print(f"  Range: [{vals.min():.3f}, {vals.max():.3f}], Mean: {vals.mean():.3f}")

corruption.head()

  Indicator: Control of Corruption: Estimate
  Shape: (6076, 4), Countries: 217, Non-null control_of_corruption: 4988
  Range: [0.106, 0.992], Mean: 0.495


,country_name,country_code,year,control_of_corruption
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,0.241659
2,Angola,AGO,1996,0.266460
3,Albania,ALB,1996,0.321219
4,Andorra,AND,1996,0.763629


## Load Employment in Agriculture

In [74]:
agriculture = load_wb_csv(
    DATA_RAW / "EmploymentInAgriculture_Historical_WorldBank_1991-2023.csv",
    "employment_agriculture"
)
agriculture.head()

  Indicator: Employment in agriculture (% of total employment) (modeled ILO estimate)
  Shape: (6076, 4), Countries: 217, Non-null employment_agriculture: 5232


,country_name,country_code,year,employment_agriculture
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,67.170502
2,Angola,AGO,1996,38.959080
3,Albania,ALB,1996,52.857857
4,Andorra,AND,1996,NaN


## Load Gini Coefficient

In [75]:
gini = load_wb_csv(
    DATA_RAW / "GiniCoefficient_Historical_WorldBank_1963-2024.csv",
    "gini"
)
gini.head()

  Indicator: Gini index
  Shape: (6076, 4), Countries: 217, Non-null gini: 1999


,country_name,country_code,year,gini
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,NaN
2,Angola,AGO,1996,NaN
3,Albania,ALB,1996,27.0
4,Andorra,AND,1996,NaN


## Load Poverty Data

We have three poverty threshold files in **different formats**:
- `poverty_3$_1963_2024.csv` — World Bank format ($3.00/day headcount ratio)
- `POV_8.30$_1963_2024.csv` — Simple CSV (country, year, headcount_ratio_830)
- `POV_10$_1963_2024.csv` — Simple CSV (country, year, headcount_ratio_1000)



In [76]:
# Poverty $3/day — World Bank format
pov3 = load_wb_csv(DATA_RAW / "poverty_3$_1963_2024.csv", "poverty_3")
print()

  Indicator: Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)
  Shape: (6076, 4), Countries: 217, Non-null poverty_3: 1999



### Load poverty $8.30 and $10 (simple CSV format)

These files use country names instead of ISO codes, so we map them using the GDP file.
Country names that don't match World Bank codes (e.g. "Lao PDR" vs "Lao People's Democratic Republic") 
are handled via `POVERTY_NAME_FIXES`. Sub-national entries and regional aggregates are dropped.

In [77]:
# Build country name → code mapping from GDP data
name_to_code = (
    gdp[["country_name", "country_code"]]
    .drop_duplicates()
    .set_index("country_name")["country_code"]
    .to_dict()
)
print(f"Name-to-code mapping: {len(name_to_code)} entries")

Name-to-code mapping: 217 entries


In [78]:
# Country name fixes for poverty CSVs (PovcalNet/WBPIP naming → WB standard)
# Sub-national entries (urban/rural) and regional/world aggregates cannot be mapped and stay dropped.
POVERTY_NAME_FIXES = {
    "Cape Verde":                           "Cabo Verde",
    "Congo":                                "Congo, Rep.",
    "Democratic Republic of Congo":         "Congo, Dem. Rep.",
    "East Timor":                           "Timor-Leste",
    "Egypt":                                "Egypt, Arab Rep.",
    "Gambia":                               "Gambia, The",
    "Iran":                                 "Iran, Islamic Rep.",
    "Ivory Coast":                          "Cote d'Ivoire",
    "Kyrgyzstan":                           "Kyrgyz Republic",
    "Laos":                                 "Lao PDR",
    "Micronesia (country)":                 "Micronesia, Fed. Sts.",
    "Palestine":                            "West Bank and Gaza",
    "Russia":                               "Russian Federation",
    "Saint Lucia":                          "St. Lucia",
    "Slovakia":                             "Slovak Republic",
    "South Korea":                          "Korea, Rep.",
    "Syria":                                "Syrian Arab Republic",
    "Turkey":                               "Turkiye",
    "Venezuela":                            "Venezuela, RB",
    "Vietnam":                              "Viet Nam",
    "Yemen":                                "Yemen, Rep.",
}

def load_simple_poverty_csv(filepath, value_col_in_file, var_name, year_start=1996, year_end=2023):
    """
    Load a simple poverty CSV (country, year, value).
    Maps country names to WB codes via POVERTY_NAME_FIXES + name_to_code.
    Sub-national (urban/rural) and regional/world aggregate entries cannot be matched and are dropped.
    """
    df = pd.read_csv(filepath)
    print(f"  Columns: {list(df.columns)}")

    df = df.rename(columns={"country": "country_name", value_col_in_file: var_name})
    df["year"] = df["year"].astype(int)
    df[var_name] = pd.to_numeric(df[var_name], errors="coerce")

    # Apply name fixes before mapping
    df["country_name_mapped"] = df["country_name"].map(lambda x: POVERTY_NAME_FIXES.get(x, x))
    df["country_code"] = df["country_name_mapped"].map(name_to_code)

    # Report unmatched — split into subnational/regional vs real countries
    unmatched = df[df["country_code"].isna()]["country_name"].unique()
    subnational_tags = ["(urban)", "(rural)", "(WB)", "World"]
    fixable = [u for u in unmatched if not any(t in u for t in subnational_tags)]
    if len(unmatched) > 0:
        print(f"  Dropped {len(unmatched)} unmatched entries "
              f"({len(unmatched) - len(fixable)} subnational/regional/world, "
              f"{len(fixable)} real countries without a fix):")
        if fixable:
            print(f"    Real countries still unmatched: {sorted(fixable)}")

    # Keep only matched rows, filter to requested year range
    df = df.dropna(subset=["country_code"])
    df = df[(df["year"] >= year_start) & (df["year"] <= year_end)]
    df = df[["country_name", "country_code", "year", var_name]].reset_index(drop=True)

    print(f"  Shape: {df.shape}, Countries: {df['country_code'].nunique()}, "
          f"Non-null {var_name}: {df[var_name].notna().sum()}")
    return df

pov830 = load_simple_poverty_csv(
    DATA_RAW / "POV_8.30$_1963_2024.csv", "headcount_ratio_830", "poverty_8_30"
)
print()

pov10 = load_simple_poverty_csv(
    DATA_RAW / "POV_10$_1963_2024.csv", "headcount_ratio_1000", "poverty_10"
)

  Columns: ['country', 'year', 'headcount_ratio_830']
  Dropped 25 unmatched entries (24 subnational/regional/world, 1 real countries without a fix):
    Real countries still unmatched: ['Taiwan']
  Shape: (1865, 4), Countries: 169, Non-null poverty_8_30: 1865

  Columns: ['country', 'year', 'headcount_ratio_1000']
  Dropped 25 unmatched entries (24 subnational/regional/world, 1 real countries without a fix):
    Real countries still unmatched: ['Taiwan']
  Shape: (1865, 4), Countries: 169, Non-null poverty_10: 1865


## Load HDI (Human Development Index)

Simple CSV from UNDP, covering 1990–2019. Missing values are coded as `..`.
Since our panel runs to 2023, we extrapolate 2020–2023 from each country's recent growth trend.

In [79]:
hdi_raw = pd.read_csv(DATA_RAW / "HDI_1990_2019.csv")
print(f"HDI raw shape: {hdi_raw.shape}")
print(f"Columns: {list(hdi_raw.columns[:5])} ... {list(hdi_raw.columns[-3:])}")
hdi_raw.head(3)

HDI raw shape: (189, 32)
Columns: ['HDI Rank', 'Country', '1990', '1991', '1992'] ... ['2017', '2018', '2019']


,HDI Rank,Country,1990,1991,1992,1993,1994,1995,1996,1997,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,169,Afghanistan,0.302,0.307,0.316,0.312,0.307,0.331,0.335,0.339,...,0.472,0.477,0.489,0.496,0.5,0.5,0.502,0.506,0.509,0.511
1,69,Albania,0.65,0.631,0.615,0.618,0.624,0.637,0.646,0.645,...,0.745,0.764,0.775,0.782,0.787,0.788,0.788,0.790,0.792,0.795
2,91,Algeria,0.572,0.576,0.582,0.586,0.59,0.595,0.602,0.611,...,0.721,0.728,0.728,0.729,0.736,0.74,0.743,0.745,0.746,0.748


In [80]:
# UNDP HDI naming → World Bank standard naming
HDI_NAME_FIXES = {
    "Bahamas":                              "Bahamas, The",
    "Bolivia (Plurinational State of)":     "Bolivia",
    "Congo":                                "Congo, Rep.",
    "Congo (Democratic Republic of the)":  "Congo, Dem. Rep.",
    "Côte d'Ivoire":                       "Cote d'Ivoire",
    "Egypt":                                "Egypt, Arab Rep.",
    "Eswatini (Kingdom of)":               "Eswatini",
    "Gambia":                               "Gambia, The",
    "Hong Kong, China (SAR)":              "Hong Kong SAR, China",
    "Iran (Islamic Republic of)":           "Iran, Islamic Rep.",
    "Korea (Democratic People's Rep. of)": "Korea, Dem. People's Rep.",
    "Korea (Republic of)":                  "Korea, Rep.",
    "Kyrgyzstan":                           "Kyrgyz Republic",
    "Lao People's Democratic Republic":    "Lao PDR",
    "Micronesia (Federated States of)":    "Micronesia, Fed. Sts.",
    "Moldova (Republic of)":               "Moldova",
    "Palestine, State of":                  "West Bank and Gaza",
    "Saint Kitts and Nevis":               "St. Kitts and Nevis",
    "Saint Lucia":                          "St. Lucia",
    "Saint Vincent and the Grenadines":    "St. Vincent and the Grenadines",
    "Slovakia":                             "Slovak Republic",
    "Tanzania (United Republic of)":       "Tanzania",
    "Turkey":                               "Turkiye",
    "Türkiye":                              "Turkiye",
    "Venezuela (Bolivarian Republic of)":  "Venezuela, RB",
    "Vietnam":                              "Viet Nam",
    "Yemen":                                "Yemen, Rep.",
    "São Tomé and Príncipe":               "Sao Tome and Principe",
}

# Drop HDI Rank, rename Country
hdi = hdi_raw.drop(columns=["HDI Rank"]).rename(columns={"Country": "country_name"})

# Melt to long format (year columns are 1990..2019)
year_cols = [c for c in hdi.columns if c != "country_name"]
hdi = hdi.melt(id_vars=["country_name"], var_name="year", value_name="hdi")

# Replace ".." with NaN, convert types
hdi["hdi"] = hdi["hdi"].replace("..", np.nan).astype(float)
hdi["year"] = hdi["year"].astype(int)

# Apply name fixes before mapping
hdi["country_name_mapped"] = hdi["country_name"].map(lambda x: HDI_NAME_FIXES.get(x, x))

# Map country names to codes
hdi["country_code"] = hdi["country_name_mapped"].map(name_to_code)

unmatched_hdi = hdi[hdi["country_code"].isna()]["country_name"].unique()
if len(unmatched_hdi) > 0:
    print(f"HDI countries not matched to WB codes: {len(unmatched_hdi)}")
    print(f"Examples: {unmatched_hdi[:10]}")
else:
    print("All HDI countries matched to WB codes.")

# Filter to 1996-2019 (we'll extrapolate 2020-2023 next)
hdi = hdi.dropna(subset=["country_code"])
hdi = hdi[hdi["year"] >= 1996].reset_index(drop=True)

print(f"HDI shape: {hdi.shape}, Countries: {hdi['country_code'].nunique()}")

All HDI countries matched to WB codes.
HDI shape: (4536, 5), Countries: 189


### Extrapolate HDI for 2020–2023

For each country we take the average annual growth rate over 2015–2019 and apply it forward. Using a geometric mean avoids overweighting outlier years.

In [81]:
def extrapolate_hdi(group):
    """
    Extrapolate HDI from 2019 to 2023 using geometric mean of recent growth.
    Growth rates are annualized to correctly handle years with missing data.
    E.g. if 2017 is missing, the 2016→2018 ratio is a 2-year rate → divide exponent by 2.
    """
    code = group["country_code"].iloc[0]
    name = group["country_name"].iloc[0]

    recent = group[group["year"].between(2015, 2019)].sort_values("year").dropna(subset=["hdi"])

    if len(recent) < 2:
        new_rows = pd.DataFrame({
            "country_name": name, "country_code": code,
            "year": [2020, 2021, 2022, 2023], "hdi": np.nan
        })
        return pd.concat([group, new_rows], ignore_index=True)

    years_arr = recent["year"].values
    vals      = recent["hdi"].values
    year_diffs = np.diff(years_arr)
    raw_ratios = vals[1:] / vals[:-1]

    annual_rates = raw_ratios ** (1.0 / year_diffs)
    avg_growth = np.exp(np.mean(np.log(annual_rates)))
    avg_growth = np.clip(avg_growth, 0.97, 1.05)

    last_year_row = group.loc[group["year"] == 2019, "hdi"]
    if len(last_year_row) == 0 or np.isnan(last_year_row.values[0]):
        last_val  = vals[-1]
        last_year = int(years_arr[-1])  # tatsächliches letztes Jahr mit Daten
    else:
        last_val  = last_year_row.values[0]
        last_year = 2019

    new_rows = []
    for yr in [2020, 2021, 2022, 2023]:
        projected = last_val * (avg_growth ** (yr - last_year))  # korrekter Exponent
        projected = min(projected, 1.0)
        new_rows.append({
            "country_name": name, "country_code": code,
            "year": yr, "hdi": projected
        })

    return pd.concat([group, pd.DataFrame(new_rows)], ignore_index=True)

hdi = hdi.groupby("country_code", group_keys=False).apply(extrapolate_hdi).reset_index(drop=True)
print(f"HDI after extrapolation: {hdi.shape}")
print(f"Year range: {hdi['year'].min()}–{hdi['year'].max()}")
print(f"HDI non-null 2020-2023: {hdi[hdi['year'] >= 2020]['hdi'].notna().sum()}")

HDI after extrapolation: (5292, 5)
Year range: 1996–2023
HDI non-null 2020-2023: 756


## Load Population from SSP Excel

To compute GDP per capita we need population figures. We use the SSP Excel file's 
Historical Reference scenario rather than a separate World Bank source.

The data is recorded every 5 years and gets filled in year by year.

In [82]:
ssp_raw = pd.read_excel(
    DATA_RAW / "GDP(Forecast)_POP_SSP_1950_2100.xlsx",
    sheet_name="data"
)
print(f"SSP raw shape: {ssp_raw.shape}")
print(f"Columns: {list(ssp_raw.columns[:8])}")
print(f"Variables: {ssp_raw['Variable'].unique()[:5]}")
print(f"Scenarios: {ssp_raw['Scenario'].unique()}")
ssp_raw.head(3)

SSP raw shape: (343259, 36)
Columns: ['Model', 'Scenario', 'Region', 'Variable', 'Unit', '1950', '1955', '1960']
Variables: ['GDP|PPP' 'Population' 'Population|Female' 'Population|Female|Age 0-4'
 'Population|Female|Age 10-14']
Scenarios: ['SSP1' 'SSP2' 'SSP3' 'SSP4' 'SSP5' 'Historical Reference']


,Model,Scenario,Region,Variable,Unit,1950,1955,1960,1965,1970,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,IIASA GDP 2023,SSP1,Afghanistan,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,327.105955,423.640014,527.886110,637.688750,745.606295,846.558705,937.135061,1018.194282,1090.567501,1127.372365
1,IIASA GDP 2023,SSP1,Africa (R10),GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,37090.900810,45893.102374,55206.857010,64686.171217,74002.250157,82862.017818,91101.053099,98692.515711,105629.637455,109475.736120
2,IIASA GDP 2023,SSP1,Albania,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,79.001085,83.924946,90.601621,96.085551,100.143830,102.783202,104.634990,105.650176,106.269104,104.419453


In [83]:
# Filter: Historical Reference, total Population
pop_ssp = ssp_raw[
    (ssp_raw["Scenario"] == "Historical Reference") &
    (ssp_raw["Variable"] == "Population")
].copy()

print(f"Population unit: {pop_ssp['Unit'].iloc[0]}")
print(f"Population rows: {len(pop_ssp)}")

# Year columns can be str ('1950'), int (1950), or float (1950.0) depending on reader
def _is_year_col(c):
    try:
        return 1900 <= int(float(str(c))) <= 2200
    except (ValueError, TypeError):
        return False

year_cols_ssp = [c for c in pop_ssp.columns if _is_year_col(c)]
print(f"Year columns found: {len(year_cols_ssp)}")
print(f"First 5: {year_cols_ssp[:5]}")
print(f"Column types: {[type(c).__name__ for c in year_cols_ssp[:3]]}")

pop_long = pop_ssp[["Region"] + year_cols_ssp].melt(
    id_vars=["Region"], var_name="year", value_name="population"
)
print(f"After melt: {pop_long.shape}, non-null pop: {pop_long['population'].notna().sum()}")

pop_long["year"] = pop_long["year"].astype(float).astype(int)
pop_long["population"] = pd.to_numeric(pop_long["population"], errors="coerce")

print(f"Pop long shape: {pop_long.shape}")
pop_long.head()

Population unit: million
Population rows: 225
Year columns found: 31
First 5: ['1950', '1955', '1960', '1965', '1970']
Column types: ['str', 'str', 'str']
After melt: (6975, 3), non-null pop: 3375
Pop long shape: (6975, 3)


,Region,year,population
0,Afghanistan,1950,7.436561
1,Africa (R10),1950,225.115119
2,Albania,1950,1.234474
3,Algeria,1950,8.892792
4,Angola,1950,4.432912


### Map SSP country names to World Bank codes

The SSP Excel uses its own country names which don't always match World Bank codes, so we map them.

In [84]:
# The SSP file uses country names in the "Region" column.
# We map these to WB country codes using the GDP-derived mapping.
# Some names differ between SSP and WB — we handle common mismatches.
#
# NOTE: "Czechia" was REMOVED — the old entry mapped it to "Czech Republic" but the WB GDP
# file uses "Czechia", so that mapping was silently preventing Czechia from matching.

SSP_NAME_FIXES = {
    "Bahamas":                                  "Bahamas, The",
    "Bolivia (Plurinational State of)":         "Bolivia",
    "China, Hong Kong SAR":                     "Hong Kong SAR, China",
    "China, Macao SAR":                         "Macao SAR, China",
    "China, Taiwan Province of China":          "Taiwan, China",
    "Congo":                                    "Congo, Rep.",
    "Côte d'Ivoire":                           "Cote d'Ivoire",
    "Curaçao":                                  "Curacao",
    "Democratic People's Republic of Korea":   "Korea, Dem. People's Rep.",
    "Democratic Republic of the Congo":         "Congo, Dem. Rep.",
    "Egypt":                                    "Egypt, Arab Rep.",
    "Gambia":                                   "Gambia, The",
    "Hong Kong":                                "Hong Kong SAR, China",
    "Iran":                                     "Iran, Islamic Rep.",
    "Iran (Islamic Republic of)":               "Iran, Islamic Rep.",
    "Kyrgyzstan":                               "Kyrgyz Republic",
    "Laos":                                     "Lao PDR",
    "Lao People's Democratic Republic":         "Lao PDR",
    "Macao":                                    "Macao SAR, China",
    "Micronesia":                               "Micronesia, Fed. Sts.",
    "Micronesia (Federated States of)":         "Micronesia, Fed. Sts.",
    "Moldova (Republic of)":                    "Moldova",
    "North Korea":                              "Korea, Dem. People's Rep.",
    "Palestine":                                "West Bank and Gaza",
    "Puerto Rico":                              "Puerto Rico (US)",
    "Republic of Korea":                        "Korea, Rep.",
    "Republic of the Congo":                    "Congo, Rep.",
    "Saint Kitts and Nevis":                    "St. Kitts and Nevis",
    "Saint Lucia":                              "St. Lucia",
    "Saint Vincent and the Grenadines":         "St. Vincent and the Grenadines",
    "Somalia":                                  "Somalia, Fed. Rep.",
    "South Korea":                              "Korea, Rep.",
    "Syria":                                    "Syrian Arab Republic",
    "São Tomé and Príncipe":                   "Sao Tome and Principe",
    "Slovakia":                                 "Slovak Republic",
    "State of Palestine":                       "West Bank and Gaza",
    "Syrian Arab Republic":                     "Syrian Arab Republic",
    "Tanzania (United Republic of)":            "Tanzania",
    "Turkey":                                   "Turkiye",
    "Türkiye":                                  "Turkiye",
    "United States Virgin Islands":             "Virgin Islands (U.S.)",
    "United States of America":                 "United States",
    "Venezuela":                                "Venezuela, RB",
    "Venezuela (Bolivarian Republic of)":       "Venezuela, RB",
    "Vietnam":                                  "Viet Nam",
    "Yemen":                                    "Yemen, Rep.",
}

# Apply name fixes
pop_long["country_name_wb"] = pop_long["Region"].map(
    lambda x: SSP_NAME_FIXES.get(x, x)
)
pop_long["country_code"] = pop_long["country_name_wb"].map(name_to_code)

# Report unmatched — distinguish real countries from regional aggregates
unmatched_pop = pop_long[pop_long["country_code"].isna()]["Region"].unique()
regional_tags = ["(R5)", "(R10)", "(R9)", "Africa", "Asia", "Europe", "European Union",
                 "Latin America", "Middle East", "OECD", "India+", "China+"]
fixable_pop = [r for r in unmatched_pop if not any(t in r for t in regional_tags)]
print(f"SSP regions not matched: {len(unmatched_pop)} "
      f"({len(unmatched_pop) - len(fixable_pop)} regional aggregates, "
      f"{len(fixable_pop)} real countries without a fix)")
if fixable_pop:
    print(f"  Real countries still unmatched: {fixable_pop}")

pop_long = pop_long.dropna(subset=["country_code"])
print(f"Matched population entries: {len(pop_long)}")

SSP regions not matched: 32 (24 regional aggregates, 8 real countries without a fix)
  Real countries still unmatched: ['French Guiana', 'Guadeloupe', 'Martinique', 'Mayotte', 'Réunion', 'Taiwan', 'Western Sahara', 'World']
Matched population entries: 5983


In [85]:
# Interpolate from 5-year intervals to annual
# Keep data from 1990 to 2025 (broader than 1996-2023 for smooth interpolation)
pop_5yr = pop_long[
    (pop_long["year"] >= 1990) & (pop_long["year"] <= 2025)
][["country_code", "year", "population"]].copy()

def interpolate_to_annual(group):
    """Interpolate 5-year population data to annual."""
    code = group["country_code"].iloc[0]
    group = group.set_index("year").sort_index()
    
    # Create annual index
    full_years = pd.RangeIndex(start=group.index.min(), stop=group.index.max() + 1)
    group = group.reindex(full_years)
    group["population"] = group["population"].interpolate(method="linear")
    group["country_code"] = code
    group.index.name = "year"
    return group.reset_index()

pop_annual = (
    pop_5yr.groupby("country_code", group_keys=False)
    .apply(interpolate_to_annual)
    .reset_index(drop=True)
)

# Filter to 1996-2023
pop_annual = pop_annual[
    (pop_annual["year"] >= 1996) & (pop_annual["year"] <= 2023)
].reset_index(drop=True)

print(f"Annual population: {pop_annual.shape}, Countries: {pop_annual['country_code'].nunique()}")
pop_annual.head()

Annual population: (5404, 3), Countries: 193


,year,country_code,population
0,1996,ABW,0.078332
1,1997,ABW,0.080779
2,1998,ABW,0.083227
3,1999,ABW,0.085674
4,2000,ABW,0.088122


## Merge all variables into one panel

Join everything on `(country_code, year)`. Start with GDP (broadest coverage), then left-join each indicator.

In [86]:
# Start with the full country-year grid from GDP
panel = gdp[["country_name", "country_code", "year"]].copy()
panel = panel.merge(gdp[["country_code", "year", "gdp"]], on=["country_code", "year"], how="left")

# Merge population
panel = panel.merge(pop_annual[["country_code", "year", "population"]], on=["country_code", "year"], how="left")

# Merge each indicator
for df, cols in [
    (corruption, ["country_code", "year", "control_of_corruption"]),
    (agriculture, ["country_code", "year", "employment_agriculture"]),
    (gini,       ["country_code", "year", "gini"]),
    (hdi,        ["country_code", "year", "hdi"]),
    (pov3,       ["country_code", "year", "poverty_3"]),
    # (pov420,   ["country_code", "year", "poverty_4_20"]),  # EXCLUDED: poverty gap, not headcount ratio
    (pov830,     ["country_code", "year", "poverty_8_30"]),
    (pov10,      ["country_code", "year", "poverty_10"]),
]:
    panel = panel.merge(df[cols], on=["country_code", "year"], how="left")

print(f"Panel shape: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}, Years: {panel['year'].min()}-{panel['year'].max()}")

# Validation: left-joins should never add or duplicate rows
expected_rows = len(gdp)
assert len(panel) == expected_rows, \
    f"Row count changed after merges: expected {expected_rows}, got {len(panel)}. Check for duplicate keys."
dupes = panel.duplicated(subset=["country_code", "year"]).sum()
assert dupes == 0, f"Found {dupes} duplicate country-year pairs after merging."
print("Merge validation passed: no duplicate rows.")

panel.head()

Panel shape: (6076, 12)
Countries: 217, Years: 1996-2023
Merge validation passed: no duplicate rows.


,country_name,country_code,year,gdp,population,control_of_corruption,employment_agriculture,gini,hdi,poverty_3,poverty_8_30,poverty_10
0,Aruba,ABW,1996,2.158919e+09,0.078332,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1996,NaN,16.791414,0.241659,67.170502,NaN,0.335,NaN,NaN,NaN
2,Angola,AGO,1996,4.735814e+10,14.170120,0.266460,38.959080,NaN,NaN,NaN,NaN,NaN
3,Albania,ALB,1996,9.675284e+09,3.271111,0.321219,52.857857,27.0,0.646,3.0,58.611904,71.59013
4,Andorra,AND,1996,1.698237e+09,NaN,0.763629,NaN,NaN,NaN,NaN,NaN,NaN


### Compute GDP per capita (PPP)

Population from the SSP file is in millions, so we scale it up before dividing. 
The result is GDP per capita in PPP terms, matching the unit used in the SSP forecasts.

In [87]:
# Check population unit from SSP file
# SSP population is in 'million' - so multiply by 1e6 before dividing
print(f"GDP example (USA 2020): {panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'gdp'].values}")
print(f"Pop example (USA 2020): {panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'population'].values}")

# Population is in millions, GDP is in current USD
panel["gdp_per_capita"] = panel["gdp"] / (panel["population"] * 1e6)

# Sanity check
print(f"\nGDP per capita (USA 2020): ${panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'gdp_per_capita'].values}")
print("(Should be ~63,000 for USA in 2020)")
print(f"\nGDP per capita non-null: {panel['gdp_per_capita'].notna().sum()}")

GDP example (USA 2020): [2.1354105e+13]
Pop example (USA 2020): [335.388236]

GDP per capita (USA 2020): $[63669.80921776]
(Should be ~63,000 for USA in 2020)

GDP per capita non-null: 5155


## Record NaN fractions before imputation

We track how much of each row was missing *before* imputation, so we can discuss data quality.

In [88]:
# Feature columns (used in modelling) and target columns (poverty)
feature_cols = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
target_cols  = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)

# Flag which gini values are real observations BEFORE any imputation.
# After imputation gini looks complete, but ~67% of values are synthetic year-medians.
# This flag lets downstream code distinguish observed from imputed gini.
panel["gini_observed"] = panel["gini"].notna()

# NaN summary BEFORE imputation
nan_before = panel[feature_cols].isna().mean()
print("=== % NaN BEFORE imputation ===")
print((nan_before * 100).round(2))
print()
print(f"Target coverage (% with real data):")
print((panel[target_cols].notna().mean() * 100).round(2))

# Per-row quality metrics (computed before imputation so they reflect true missingness)
panel["imputation_pct"]    = panel[feature_cols].isna().mean(axis=1)   # feature missingness
panel["target_missing_pct"] = panel[target_cols].isna().mean(axis=1)   # target missingness

=== % NaN BEFORE imputation ===
gdp_per_capita            15.16
hdi                       17.05
control_of_corruption     17.91
employment_agriculture    13.89
gini                      67.10
dtype: float64

Target coverage (% with real data):
poverty_3       32.90
poverty_8_30    30.69
poverty_10      30.69
dtype: float64


## Imputation (features only)

Poverty targets are not imputed — we only train on real observations.

For features, we fill gaps per country in four steps:
1. Linear interpolation between known values
2. Forward/backward fill (max 2 years each)
3. Year median across all countries
4. Nearest-year median as last resort

No countries are dropped for missing data, since poorer countries tend to have less coverage.


In [89]:
# Sort for proper interpolation
panel = panel.sort_values(["country_code", "year"]).reset_index(drop=True)

# Only impute FEATURES, not targets — targets should retain real observed values only
# so that the ML model trains on actual poverty data, not synthetic medians.
for col in feature_cols:
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
    )
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.ffill(limit=2))
    )
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.bfill(limit=2))
    )

print("After interpolation + ffill/bfill (features only):")
print((panel[feature_cols].isna().mean() * 100).round(2))

After interpolation + ffill/bfill (features only):
gdp_per_capita            14.78
hdi                       15.70
control_of_corruption      8.08
employment_agriculture    13.82
gini                      36.93
dtype: float64


In [90]:
# Step 3: Fill remaining NaN with year-median (FEATURES ONLY)
# This preserves temporal structure — a missing value in 1996 gets the 1996 median,
# not a global median that blends 1996–2023 together.
for col in feature_cols:
    year_medians = panel.groupby("year")[col].transform("median")
    panel[col] = panel[col].fillna(year_medians)

# Final fallback: nearest-year median (for any year that has no data at all)
for col in feature_cols:
    if panel[col].isna().any():
        yr_med = panel.groupby("year")[col].median()
        yr_med = yr_med.interpolate(method="linear").ffill().bfill()
        panel[col] = panel[col].fillna(panel["year"].map(yr_med))

print("After year-median fill (features only):")
print((panel[feature_cols].isna().mean() * 100).round(2))

After year-median fill (features only):
gdp_per_capita            0.0
hdi                       0.0
control_of_corruption     0.0
employment_agriculture    0.0
gini                      0.0
dtype: float64


## Save output

In [91]:
# Final column selection and order
output_cols = [
    "country_name", "country_code", "year",
    "gdp_per_capita", "hdi", "control_of_corruption",
    "employment_agriculture", "gini", "gini_observed",
    "poverty_3", "poverty_8_30", "poverty_10",   # poverty_4_20 excluded
    "imputation_pct", "target_missing_pct",
]
panel = panel[output_cols]

panel.to_csv(DATA_PROCESSED / "historical_panel.csv", index=False)
print(f"Saved: {DATA_PROCESSED / 'historical_panel.csv'}")
print(f"Shape: {panel.shape}")
print(f"\nColumns: {list(panel.columns)}")

Saved: ../data/processed/historical_panel.csv
Shape: (6076, 14)

Columns: ['country_name', 'country_code', 'year', 'gdp_per_capita', 'hdi', 'control_of_corruption', 'employment_agriculture', 'gini', 'gini_observed', 'poverty_3', 'poverty_8_30', 'poverty_10', 'imputation_pct', 'target_missing_pct']


## Summary

In [92]:
print("=" * 60)
print("HISTORICAL PANEL SUMMARY")
print("=" * 60)
print(f"Countries:  {panel['country_code'].nunique()}")
print(f"Year range: {panel['year'].min()} – {panel['year'].max()}")
print(f"Total rows: {len(panel)}")
print()
print("% NaN BEFORE imputation (features):")
print((nan_before * 100).round(2))
print()
print("% NaN AFTER imputation (features):")
print((panel[feature_cols].isna().mean() * 100).round(2))
print()
print(f"Gini: {panel['gini_observed'].sum()} real observations, "
      f"{(~panel['gini_observed']).sum()} imputed ({(~panel['gini_observed']).mean()*100:.1f}%)")
print()
print("Average imputation_pct per row:", panel["imputation_pct"].mean().round(4))
print("Average target_missing_pct per row:", panel["target_missing_pct"].mean().round(4))
print()
print(panel.describe().round(3))

HISTORICAL PANEL SUMMARY
Countries:  217
Year range: 1996 – 2023
Total rows: 6076

% NaN BEFORE imputation (features):
gdp_per_capita            15.16
hdi                       17.05
control_of_corruption     17.91
employment_agriculture    13.89
gini                      67.10
dtype: float64

% NaN AFTER imputation (features):
gdp_per_capita            0.0
hdi                       0.0
control_of_corruption     0.0
employment_agriculture    0.0
gini                      0.0
dtype: float64

Gini: 1999 real observations, 4077 imputed (67.1%)

Average imputation_pct per row: 0.2622
Average target_missing_pct per row: 0.6857

           year  gdp_per_capita       hdi  control_of_corruption  \
count  6076.000        6076.000  6076.000               6076.000   
mean   2009.500       16581.727     0.685                  0.492   
std       8.078       19793.637     0.150                  0.192   
min    1996.000         280.941     0.244                  0.106   
25%    2002.750        4068.8

---
**Output produced:** `data/processed/historical_panel.csv`  
Contains one row per country-year (1996–2023) with 5 features, 3 poverty targets, and imputation quality flag.